<div style="direction: rtl; line-height: 1.8;">
<h1>NB12g — بازسازی تمیز و آزمایشی Golden Dataset</h1>
<p>این نوت‌بوک مسیر اصلی مرحلهٔ ۱۲ است و از نوت‌بوک‌های 12 تا 12f و آزمایش 12g قبلی، فقط artifactهای معتبر را می‌خواند. هیچ نسخهٔ جدیدی از Golden Dataset منتشر یا فعال نمی‌شود.</p>
<p>هدف این اجرا ساخت یک آزمایش قابل‌بررسی است: هر سؤال از یک quote دقیق در chunk واقعی تولید می‌شود، پاسخ نیز span دقیق همان quote است. بنابراین هیچ سؤال آزاد، answer آزاد یا قالب عمومی وارد آزمایش نمی‌شود.</p>
</div>


<div style="direction: rtl; line-height: 1.8;">
<h2>۱. قرارداد اجرا</h2>
<ul>
<li>Golden Dataset v1 فقط خوانده می‌شود.</li>
<li>نام هیچ خروجی با <code>golden_dataset_vN.jsonl</code> شروع نمی‌شود.</li>
<li>NB13 فعلی پس از اجرای این notebook همچنان v1 را می‌خواند.</li>
<li>همهٔ cacheها با hash محتوا شناخته می‌شوند، نه نام notebook یا state kernel.</li>
</ul>
</div>


In [1]:
# Code Cell 01
from pathlib import Path
from collections import Counter, defaultdict
from datetime import datetime, timezone
from urllib.parse import urlparse
import hashlib, importlib.util, json, math, os, random, re, time, uuid
import numpy as np

SOURCE_DATASET_FILENAME = "golden_dataset_v1.jsonl"
EXPERIMENT_NAME = "golden_dataset_quote_grounded_experiment"
QUESTION_PROMPT_VERSION = "nb12g_clean_quote_question_v1"
QUESTIONS_PER_QUOTE = 1
MAX_QUOTES_PER_SEED = 3
RANDOM_SEED = 14050621
random.seed(RANDOM_SEED)

cwd = Path.cwd().resolve()
project_root = next((p for p in (cwd, *cwd.parents)
                     if (p / "notebooks" / "phase2").is_dir() and (p / "data" / "phase2").is_dir()), None)
assert project_root is not None, "Open this notebook from inside the RAG_finance repository."

golden_dir = project_root / "notebooks" / "phase2" / "golden_dataset"
v1_path = golden_dir / SOURCE_DATASET_FILENAME
assert v1_path.is_file(), f"Missing source dataset: {v1_path}"
print("Project root:", project_root)
print("Read-only Golden input:", v1_path)
print("Active-dataset creation:", False)
print("\nEXECUTION CONTRACT: PASS")


Project root: /Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/RAG_finance
Read-only Golden input: /Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/RAG_finance/notebooks/phase2/golden_dataset/golden_dataset_v1.jsonl
Active-dataset creation: False

EXECUTION CONTRACT: PASS


In [2]:
# Code Cell 02
def sha256_file(path):
    h = hashlib.sha256()
    with path.open("rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

def read_json(path):
    return json.loads(path.read_text(encoding="utf-8"))

def read_jsonl(path):
    rows=[]
    with path.open(encoding="utf-8") as f:
        for line_no,line in enumerate(f,1):
            if line.strip():
                try: row=json.loads(line)
                except json.JSONDecodeError as e: raise AssertionError(f"Invalid JSONL {path}:{line_no}") from e
                assert isinstance(row,dict)
                rows.append(row)
    return rows

def write_json_atomic(path, value):
    tmp=path.with_suffix(path.suffix+".tmp")
    tmp.write_text(json.dumps(value,ensure_ascii=False,indent=2),encoding="utf-8")
    tmp.replace(path)

def write_jsonl_atomic(path, rows):
    tmp=path.with_suffix(path.suffix+".tmp")
    with tmp.open("w",encoding="utf-8") as f:
        for row in rows: f.write(json.dumps(row,ensure_ascii=False)+"\n")
    tmp.replace(path)

def canon(value):
    value=str(value or "").translate(str.maketrans({"ي":"ی","ى":"ی","ك":"ک","ة":"ه","ۀ":"ه","‌":" ","‏":" ","‎":" "})).lower()
    return re.sub(r"\s+", " ", value).strip()

def words(value):
    return re.findall(r"[\w\u0600-\u06ff]+",canon(value))

STOP={"از","به","در","با","برای","که","این","آن","را","و","یا","یک","می","شود","شده","است","هست","بود","باشد","های","ها","خود","چه","چرا","چگونه","درباره","بر","اگر","اما","هم"}
def content_words(value): return {w for w in words(value) if len(w)>2 and w not in STOP}

def parse_json_object(raw):
    raw=(raw or "").strip()
    raw=re.sub(r"^```(?:json)?\s*", "", raw)
    raw=re.sub(r"\s*```$", "", raw)
    dec=json.JSONDecoder()
    for i,c in enumerate(raw):
        if c=="{":
            try:
                item,_=dec.raw_decode(raw[i:])
                if isinstance(item,dict): return item
            except json.JSONDecodeError: pass
    return None

print("Utilities loaded.\nUTILITIES: PASS")


Utilities loaded.
UTILITIES: PASS


<div style="direction: rtl; line-height: 1.8;"><h2>۲. کنترل NB11 و استخراج seedهای معتبر</h2><p>این بخش به state notebookهای قبلی وابسته نیست و hash artifactهای freezeشده را کنترل می‌کند.</p></div>


In [3]:
# Code Cell 03
manifest_paths=sorted(project_root.rglob("evaluation_freeze_manifest_v1.json"))
assert len(manifest_paths)==1, f"Expected one NB11 manifest, found {len(manifest_paths)}"
manifest_path=manifest_paths[0]
manifest=read_json(manifest_path)
assert manifest.get("status")=="FROZEN_FOR_EVALUATION"
artifacts=manifest["artifacts"]
bad=[]
for name,d in artifacts.items():
    p=project_root/d["path"]
    if not p.is_file() or p.stat().st_size != d.get("bytes") or sha256_file(p) != d.get("sha256"):
        bad.append(name)
assert not bad, f"NB11 frozen artifacts failed: {bad}"

child_desc=artifacts["parent_child_children"]
parent_desc=artifacts["parent_child_parents"]
medium_desc=artifacts["structure_aware_medium"]
child_path=project_root/child_desc["path"]
parent_path=project_root/parent_desc["path"]
medium_path=project_root/medium_desc["path"]
print("NB11 manifest:",manifest_path.relative_to(project_root))
print("Validated frozen artifacts:",len(artifacts))
print("\nNB11 FREEZE: PASS")


NB11 manifest: reports/phase2/chunking/evaluation_freeze_manifest_v1.json
Validated frozen artifacts: 15

NB11 FREEZE: PASS


In [4]:
# Code Cell 04
v1=read_jsonl(v1_path)
required={"candidate_id","question","expected_answer","source_seed_id","source_lineage","source_evidence","taxonomy_topics","competency_ids"}
assert all(required <= set(r) for r in v1), "v1 schema is incomplete"
by_seed=defaultdict(list)
for r in v1: by_seed[int(r["source_seed_id"])].append(r)
assert sorted(by_seed)==list(range(1,29))

seeds={}
for seed_id,records in sorted(by_seed.items()):
    evidences={canon(r["source_evidence"]):r["source_evidence"] for r in records}
    answers={canon(r["expected_answer"]):r["expected_answer"] for r in records}
    assert len(evidences)==1 and len(answers)==1
    seeds[seed_id]={
        "seed_id":seed_id,
        "lineage":f"pilot_{seed_id}",
        "approved_evidence":next(iter(evidences.values())),
        "approved_answer":next(iter(answers.values())),
        "taxonomy_topics":sorted({x for r in records for x in r["taxonomy_topics"]}),
        "competency_ids":sorted({x for r in records for x in r["competency_ids"]}),
        "legacy_questions":[r["question"] for r in records],
    }
def seed_split(seed_id):
    bucket=int.from_bytes(hashlib.sha256(f"nb12g-clean-split:{seed_id}".encode()).digest()[:4],"big") % 100
    return "train" if bucket < 60 else "dev" if bucket < 80 else "test"
split_by_seed={seed_id:seed_split(seed_id) for seed_id in seeds}
assert set(split_by_seed.values()) == {"train","dev","test"}
print("v1 records:",len(v1),"| seeds:",len(seeds),"| v1 SHA:",sha256_file(v1_path))
print("Seed-level split:",dict(Counter(split_by_seed.values())))
print("Legacy questions used for generation:",False)
print("\nSEED FREEZE: PASS")


v1 records: 573 | seeds: 28 | v1 SHA: df18bd384261400986e2c3bcdeda0cffa084243fb99f073c51396dbe2d9f2729
Seed-level split: {'train': 19, 'dev': 3, 'test': 6}
Legacy questions used for generation: False

SEED FREEZE: PASS


<div style="direction: rtl; line-height: 1.8;"><h2>۳. provenance و quoteهای واقعی</h2><p>برخلاف آزمایش قبلی، quoteها قبل از تولید سؤال و به‌شکل deterministic از chunk واقعی انتخاب می‌شوند. مدل اجازه ندارد quote بسازد یا تغییر دهد.</p></div>


In [5]:
# Code Cell 05
registry_paths = [
    path
    for path in project_root.rglob("human_review_registry.json")
    if "before_" not in path.name
]
assert registry_paths, "Human-review registry is missing."
registry_path = max(registry_paths, key=lambda path: path.stat().st_mtime)

source_packet_paths = sorted(
    project_root.rglob("source_packet_pilots23_28_v1.json")
)
assert len(source_packet_paths) == 1, (
    "Expected exactly one source packet for pilots 23–28."
)

registry = read_json(registry_path)
source_packet = read_json(source_packet_paths[0])
completed_cases = registry.get("completed_human_review_cases", [])
source_packets = {
    int(packet["reference_pilot_number"]): packet
    for packet in source_packet.get("pilots", [])
}

assert len(completed_cases) == 22
assert sorted(source_packets) == list(range(23, 29))

provenance = {}

for seed_id, review_case in enumerate(completed_cases, 1):
    document_ids = sorted(
        {
            item.get("document_id")
            for item in review_case.get("source_evidence", [])
            if isinstance(item, dict) and item.get("document_id")
        }
    )

    provenance[seed_id] = {
        "pilot_case_id": review_case["pilot_case_id"],
        "source": "human_review_registry",
        "document_ids": document_ids,
        "raw": review_case,
    }

for seed_id in range(23, 29):
    packet = source_packets[seed_id]
    claim_document_ids = {
        claim.get("content_source", {}).get("document_id")
        for claim in packet.get("claims", [])
        if isinstance(claim.get("content_source"), dict)
    }

    provenance[seed_id] = {
        "pilot_case_id": packet["pilot_case_id"],
        "source": "source_packet",
        "document_ids": sorted(
            {
                document_id
                for document_id in (
                    set(packet.get("accepted_website_document_ids", []))
                    | claim_document_ids
                )
                if document_id
            }
        ),
        "raw": packet,
    }

assert sorted(provenance) == list(range(1, 29))

children = read_jsonl(child_path)
parents = read_jsonl(parent_path)
medium = read_jsonl(medium_path)
parent_by_id = {record["chunk_id"]: record for record in parents}

child_by_document = defaultdict(list)

for chunk in children:
    identifiers = set(chunk.get("member_document_ids", []))

    if chunk.get("source_id"):
        identifiers.add(chunk["source_id"])

    for document_id in identifiers:
        child_by_document[document_id].append(chunk)

def coverage(reference, text):
    reference_words = content_words(reference)

    if not reference_words:
        return 0.0

    return len(reference_words & content_words(text)) / len(reference_words)

source_chunks = {}

for seed_id, seed in seeds.items():
    matching_chunks = []
    seen_chunk_ids = set()

    for document_id in provenance[seed_id]["document_ids"]:
        for chunk in child_by_document.get(document_id, []):
            if chunk["chunk_id"] in seen_chunk_ids:
                continue

            seen_chunk_ids.add(chunk["chunk_id"])
            matching_chunks.append(chunk)

    linkage_basis = "REGISTERED_DOCUMENT_ID"

    if not matching_chunks:
        reviewed_evidence_units = [
            canon(item.get("text", ""))
            for item in provenance[seed_id]["raw"].get(
                "source_evidence", []
            )
            if isinstance(item, dict)
            and len(canon(item.get("text", ""))) >= 100
        ]

        matching_chunks = [
            chunk
            for chunk in medium
            if any(
                evidence_unit in canon(chunk["content"])
                for evidence_unit in reviewed_evidence_units
            )
        ]
        linkage_basis = "EXACT_REVIEWED_EVIDENCE_FALLBACK"

    assert matching_chunks, f"No source chunks found for seed {seed_id}."

    ranked_chunks = sorted(
        (
            (coverage(seed["approved_evidence"], chunk["content"]), chunk)
            for chunk in matching_chunks
        ),
        key=lambda item: (-item[0], item[1]["chunk_id"]),
    )

    best_coverage = ranked_chunks[0][0]
    relative_threshold = best_coverage * 0.72
    selected_chunks = [
        chunk
        for score, chunk in ranked_chunks
        if score >= relative_threshold
    ][:4]

    # Always preserve the best direct-provenance chunk.
    if not selected_chunks:
        selected_chunks = [ranked_chunks[0][1]]

    source_chunks[seed_id] = {
        "basis": linkage_basis,
        "selected": selected_chunks,
        "matching": [chunk for _, chunk in ranked_chunks],
        "best_coverage": float(best_coverage),
    }

print("Seeds with provenance:", len(source_chunks))
print(
    "Weak lexical links:",
    [
        seed_id
        for seed_id, result in source_chunks.items()
        if result["best_coverage"] < 0.18
    ],
)
print("\nPROVENANCE AND SOURCE CHUNKS: PASS")


Seeds with provenance: 28
Weak lexical links: [22]

PROVENANCE AND SOURCE CHUNKS: PASS


<div style="direction: rtl; line-height: 1.8;">
<h2>۳. انتخاب quote واقعی و ترمیم کنترل‌شده</h2>
<p>ابتدا جمله‌های دقیق از chunkهای متصل به provenance انتخاب می‌شوند. اگر یک جمله به‌تنهایی کافی نباشد، پنجره‌های یک تا سه‌جمله‌ای از همان منبع مستقیم بررسی می‌شوند. هیچ متن ساختگی و هیچ evidence قدیمی به‌عنوان quote جایگزین نمی‌شود.</p>
</div>


In [6]:
# Code Cell 06
SINGLE_SENTENCE_MIN_COVERAGE = 0.15
DIRECT_WINDOW_MIN_COVERAGE = 0.16

def sentences(text):
    parts = re.split(r"(?<=[.!؟])\s+|\n+", str(text))
    return [
        re.sub(r"\s+", " ", part).strip()
        for part in parts
        if len(words(part)) >= 6
    ]

def sentence_windows(text, maximum_sentences=3):
    source_sentences = sentences(text)
    windows = []

    for start_index in range(len(source_sentences)):
        for width in range(1, maximum_sentences + 1):
            end_index = start_index + width

            if end_index > len(source_sentences):
                continue

            window = " ".join(
                source_sentences[start_index:end_index]
            ).strip()

            if 8 <= len(words(window)) <= 110:
                windows.append(window)

    return windows

def quote_record(seed_id, quote_index, score, quote, chunk, basis):
    return {
        "quote_id": f"seed{seed_id:02d}_quote{quote_index:02d}",
        "source_seed_id": seed_id,
        "quote": quote,
        "quote_hash": hashlib.sha256(
            canon(quote).encode("utf-8")
        ).hexdigest(),
        "quote_coverage": float(score),
        "chunk_id": chunk["chunk_id"],
        "parent_chunk_id": chunk.get("parent_chunk_id"),
        "label_basis": basis,
    }

quote_ledger = []
initially_unresolved_seed_ids = []

for seed_id, seed in seeds.items():
    candidates = []

    for chunk in source_chunks[seed_id]["selected"]:
        for sentence in sentences(chunk["content"]):
            score = coverage(seed["approved_evidence"], sentence)

            if score >= SINGLE_SENTENCE_MIN_COVERAGE:
                candidates.append((score, sentence, chunk))

    candidates.sort(
        key=lambda item: (
            -item[0],
            -len(words(item[1])),
            item[2]["chunk_id"],
        )
    )

    chosen = []
    seen_quotes = set()

    for score, quote, chunk in candidates:
        canonical_quote = canon(quote)

        if canonical_quote in seen_quotes:
            continue

        seen_quotes.add(canonical_quote)
        chosen.append((score, quote, chunk))

        if len(chosen) == MAX_QUOTES_PER_SEED:
            break

    if not chosen:
        initially_unresolved_seed_ids.append(seed_id)
        continue

    for quote_index, (score, quote, chunk) in enumerate(chosen, 1):
        quote_ledger.append(
            quote_record(
                seed_id,
                quote_index,
                score,
                quote,
                chunk,
                source_chunks[seed_id]["basis"],
            )
        )

print("Initial exact-source quotes:", len(quote_ledger))
print("Initially unresolved seeds:", initially_unresolved_seed_ids)
print("\nINITIAL QUOTE SELECTION: PASS")


Initial exact-source quotes: 44
Initially unresolved seeds: [1, 7, 11, 14, 20, 22, 26, 27, 28]

INITIAL QUOTE SELECTION: PASS


In [7]:
# Code Cell 07
def is_direct_provenance_chunk(chunk, registered_document_ids):
    chunk_document_ids = set(chunk.get("member_document_ids", []))

    if chunk.get("source_id"):
        chunk_document_ids.add(chunk["source_id"])

    return bool(set(registered_document_ids) & chunk_document_ids)

all_frozen_retrieval_chunks = []
seen_frozen_chunk_ids = set()

for chunk in [*children, *medium]:
    if chunk["chunk_id"] in seen_frozen_chunk_ids:
        continue

    seen_frozen_chunk_ids.add(chunk["chunk_id"])
    all_frozen_retrieval_chunks.append(chunk)

resolved_quote_windows = []
source_review_queue = []

for seed_id in initially_unresolved_seed_ids:
    seed = seeds[seed_id]
    registered_document_ids = provenance[seed_id]["document_ids"]
    window_candidates = []

    direct_chunks = [
        chunk
        for chunk in all_frozen_retrieval_chunks
        if is_direct_provenance_chunk(
            chunk,
            registered_document_ids,
        )
    ]

    for chunk in direct_chunks:
        for window in sentence_windows(chunk["content"]):
            window_candidates.append(
                (
                    coverage(seed["approved_evidence"], window),
                    window,
                    chunk,
                )
            )

    window_candidates.sort(
        key=lambda item: (
            -item[0],
            not item[2]["chunk_id"].startswith(
                "chunk:parent_child:child:"
            ),
            -len(words(item[1])),
            item[2]["chunk_id"],
        )
    )

    eligible_windows = [
        candidate
        for candidate in window_candidates
        if candidate[0] >= DIRECT_WINDOW_MIN_COVERAGE
    ]

    if eligible_windows:
        score, quote, chunk = eligible_windows[0]
        resolved_quote_windows.append(
            quote_record(
                seed_id,
                1,
                score,
                quote,
                chunk,
                "DIRECT_PROVENANCE_WINDOW_REPAIR",
            )
        )
        continue

    best_candidate = window_candidates[0] if window_candidates else None
    source_review_queue.append(
        {
            "source_seed_id": seed_id,
            "pilot_case_id": provenance[seed_id]["pilot_case_id"],
            "quote_id": None,
            "quote": best_candidate[1] if best_candidate else None,
            "question": None,
            "answer_span": None,
            "review_reasons": [
                "direct_source_quote_requires_content_adjudication"
            ],
            "best_direct_window_coverage": (
                float(best_candidate[0]) if best_candidate else 0.0
            ),
            "best_direct_chunk_id": (
                best_candidate[2]["chunk_id"]
                if best_candidate
                else None
            ),
            "human_decision": None,
            "human_notes": None,
        }
    )

all_quote_ledger = [*quote_ledger, *resolved_quote_windows]
all_quote_ledger.sort(
    key=lambda item: (
        item["source_seed_id"],
        item["quote_id"],
    )
)

assert len(
    {item["quote_hash"] for item in all_quote_ledger}
) == len(all_quote_ledger), "Duplicate quote texts were selected."

resolved_seed_ids = sorted(
    {item["source_seed_id"] for item in resolved_quote_windows}
)
manual_review_seed_ids = sorted(
    {item["source_seed_id"] for item in source_review_queue}
)

assert not (set(resolved_seed_ids) & set(manual_review_seed_ids))
assert sorted(
    set(resolved_seed_ids) | set(manual_review_seed_ids)
) == sorted(initially_unresolved_seed_ids)

print("Window-resolved seeds:", resolved_seed_ids)
print("Seeds remaining in source review:", manual_review_seed_ids)
print("Final quote candidates:", len(all_quote_ledger))
print("\nDIRECT QUOTE-WINDOW RESOLUTION: PASS")


Window-resolved seeds: [1, 11, 14, 20, 26, 27, 28]
Seeds remaining in source review: [7, 22]
Final quote candidates: 51

DIRECT QUOTE-WINDOW RESOLUTION: PASS


<div style="direction: rtl; line-height: 1.8;">
<h2>۴. تولید سؤال و پاسخ کوتاه با cache پایدار</h2>
<p>cache با hash دیتاست، نسخهٔ prompt، مدل و hash خود quote شناخته می‌شود؛ بنابراین تغییر نام notebook یا Restart/Run All باعث اجرای دوبارهٔ درخواست‌های موفق قبلی نمی‌شود.</p>
</div>


In [8]:
# Code Cell 08
from dotenv import load_dotenv
from openai import OpenAI

assert importlib.util.find_spec("openai") is not None
assert importlib.util.find_spec("dotenv") is not None

environment_path = project_root / ".env"
assert environment_path.is_file(), "Project .env file is missing."
load_dotenv(environment_path, override=False)

api_key = os.getenv("ARVAN_AI_API_KEY")
chat_url = os.getenv("ARVAN_CHAT_URL")
chat_model = os.getenv("ARVAN_CHAT_MODEL")

assert api_key
assert chat_url
assert chat_model
assert urlparse(chat_url).scheme == "https"

chat = OpenAI(
    api_key=api_key,
    base_url=chat_url,
    timeout=180,
    max_retries=1,
)

dataset_sha256 = sha256_file(v1_path)
manifest_sha256 = sha256_file(manifest_path)
cache_namespace_payload = {
    "dataset_sha256": dataset_sha256,
    "prompt_version": QUESTION_PROMPT_VERSION,
    "model": chat_model,
}
cache_namespace = hashlib.sha256(
    json.dumps(
        cache_namespace_payload,
        sort_keys=True,
    ).encode("utf-8")
).hexdigest()[:24]

artifact_root = (
    project_root
    / "artifacts"
    / "phase2"
    / "golden_dataset_rebuild_clean"
)
shared_cache_directory = (
    artifact_root
    / "shared_question_cache"
    / cache_namespace
)
shared_cache_directory.mkdir(parents=True, exist_ok=True)

build_identity = {
    "v1_sha256": dataset_sha256,
    "nb11_manifest_sha256": manifest_sha256,
    "prompt_version": QUESTION_PROMPT_VERSION,
    "model": chat_model,
    "quote_hashes": [
        item["quote_hash"]
        for item in all_quote_ledger
    ],
}
build_fingerprint = hashlib.sha256(
    json.dumps(build_identity, sort_keys=True).encode("utf-8")
).hexdigest()

run_directory = artifact_root / "runs" / build_fingerprint
run_directory.mkdir(parents=True, exist_ok=True)

legacy_cache_directories = sorted(
    path
    for path in artifact_root.glob("*/question_cache")
    if path.is_dir()
)

print("Chat model:", chat_model)
print("Build fingerprint:", build_fingerprint)
print("Stable cache namespace:", cache_namespace)
print("Legacy cache directories discovered:", len(legacy_cache_directories))
print("\nMODEL AND CACHE PREFLIGHT: PASS")


Chat model: Qwen3-30B-A3B
Build fingerprint: a0f3cc56c178ee5ab4a8d7229d5747eff347fd72cbac46ba29e06fe95c1ca161
Stable cache namespace: 4bf4113103410cf667037ff8
Legacy cache directories discovered: 1

MODEL AND CACHE PREFLIGHT: PASS


In [9]:
# Code Cell 09
FORBIDDEN_QUESTION_PATTERNS = (
    "طبق متن",
    "بر اساس متن",
    "در این متن",
    "در این بخش",
    "منبع",
    "کدام یک از موارد زیر",
    "چه نکته ای باید بررسی شود",
    "چه مفهومی باید درک شود",
)
ALLOWED_INTENTS = (
    "DIRECT_FACT",
    "EXPLANATION",
    "CONSEQUENCE",
    "COMPARISON",
)

def prompt_for_quote(item):
    return f"""
فقط بر پایهٔ QUOTE زیر یک پرسش فارسی طبیعی و مستقل بساز.
پاسخ باید عبارتی کوتاه باشد که عیناً و بدون تغییر در QUOTE وجود دارد.
هیچ دانش، اصطلاح، مثال، عدد یا موضوع بیرون از QUOTE اضافه نکن.
از عبارت‌های «طبق متن»، «در این بخش» و «منبع» استفاده نکن.
سؤال تستی بدون گزینه نساز.
اگر quote برای پرسشی روشن مناسب نیست، usable را false بگذار.
خروجی فقط یک JSON معتبر باشد:
{{"usable":true,"intent":"DIRECT_FACT|EXPLANATION|CONSEQUENCE|COMPARISON","question":"...؟","answer_span":"..."}}

QUOTE:
{item['quote']}
""".strip()

def validate_payload(payload, item):
    errors = []

    if not isinstance(payload, dict):
        return ["payload_not_object"]

    if payload.get("usable") is not True:
        return ["model_marked_not_usable"]

    question = str(payload.get("question", "")).strip()
    answer = str(payload.get("answer_span", "")).strip()

    if payload.get("intent") not in ALLOWED_INTENTS:
        errors.append("invalid_intent")

    if not question.endswith("؟"):
        errors.append("question_mark")

    if not 4 <= len(words(question)) <= 40:
        errors.append("question_length")

    if any(
        canon(pattern) in canon(question)
        for pattern in FORBIDDEN_QUESTION_PATTERNS
    ):
        errors.append("generic_or_invalid_form")

    if not 1 <= len(words(answer)) <= 28:
        errors.append("answer_length")

    if canon(answer) not in canon(item["quote"]):
        errors.append("answer_not_exact_span")

    if not (content_words(question) & content_words(item["quote"])):
        errors.append("question_has_no_quote_signal")

    return sorted(set(errors))

def call_question(prompt):
    response = chat.chat.completions.create(
        model=chat_model,
        messages=[
            {
                "role": "system",
                "content": "Return only one valid JSON object.",
            },
            {
                "role": "user",
                "content": prompt,
            },
        ],
        temperature=0.1,
        max_tokens=800,
    )

    assert response.choices
    raw_response = (
        response.choices[0].message.content or ""
    ).strip()

    return (
        raw_response,
        parse_json_object(raw_response),
        response.choices[0].finish_reason,
    )

def discover_cached_record(quote_hash):
    canonical_path = shared_cache_directory / f"{quote_hash}.json"
    candidate_paths = [canonical_path]
    candidate_paths.extend(
        directory / f"{quote_hash}.json"
        for directory in legacy_cache_directories
    )

    for candidate_path in candidate_paths:
        if candidate_path.is_file():
            return candidate_path, read_json(candidate_path)

    return canonical_path, None

generation = []
cache_reuse_count = 0
live_request_count = 0

print("=== RESUMABLE QUOTE-GROUNDED GENERATION ===")

for position, item in enumerate(all_quote_ledger, 1):
    cache_path, generation_record = discover_cached_record(
        item["quote_hash"]
    )

    cached_errors = (
        validate_payload(generation_record.get("payload"), item)
        if isinstance(generation_record, dict)
        else ["no_cache"]
    )

    if not cached_errors:
        cache_reuse_count += 1
        canonical_cache_path = (
            shared_cache_directory / f"{item['quote_hash']}.json"
        )

        if cache_path != canonical_cache_path:
            write_json_atomic(canonical_cache_path, generation_record)
    else:
        base_prompt = prompt_for_quote(item)
        current_errors = cached_errors

        for attempt in range(1, 4):
            retry_prompt = base_prompt

            if attempt > 1:
                retry_prompt += (
                    "\nخطاهای پاسخ قبلی: "
                    + json.dumps(current_errors, ensure_ascii=False)
                    + "\nفقط JSON معتبر و کاملاً مبتنی بر quote بده."
                )

            try:
                raw_response, payload, finish_reason = call_question(
                    retry_prompt
                )
                live_request_count += 1
                current_errors = validate_payload(payload, item)
            except Exception as error:
                live_request_count += 1
                raw_response = ""
                payload = None
                finish_reason = None
                current_errors = [
                    "model_request_error:" + type(error).__name__
                ]
            generation_record = {
                "quote_id": item["quote_id"],
                "quote_hash": item["quote_hash"],
                "attempt": attempt,
                "model": chat_model,
                "payload": payload,
                "raw_response": raw_response,
                "finish_reason": finish_reason,
                "errors": current_errors,
            }
            write_json_atomic(
                shared_cache_directory / f"{item['quote_hash']}.json",
                generation_record,
            )

            if not current_errors:
                break

    generation.append({**item, **generation_record})
    status = "PASS" if not generation_record["errors"] else "REVIEW"
    print(
        f"Quote {position}/{len(all_quote_ledger)} | "
        f"seed={item['source_seed_id']} | {status}"
    )

assert len(generation) == len(all_quote_ledger)

print("\nCached successful generations reused:", cache_reuse_count)
print("Live model requests:", live_request_count)
print(
    "Machine-valid generations:",
    sum(not item["errors"] for item in generation),
)
print(
    "Generation review items:",
    sum(bool(item["errors"]) for item in generation),
)
print("\nQUOTE-GROUNDED GENERATION: COMPLETE")


=== RESUMABLE QUOTE-GROUNDED GENERATION ===
Quote 1/51 | seed=1 | PASS
Quote 2/51 | seed=2 | PASS
Quote 3/51 | seed=2 | PASS
Quote 4/51 | seed=3 | PASS
Quote 5/51 | seed=3 | PASS
Quote 6/51 | seed=3 | PASS
Quote 7/51 | seed=4 | PASS
Quote 8/51 | seed=4 | PASS
Quote 9/51 | seed=4 | PASS
Quote 10/51 | seed=5 | PASS
Quote 11/51 | seed=5 | PASS
Quote 12/51 | seed=6 | PASS
Quote 13/51 | seed=6 | PASS
Quote 14/51 | seed=8 | PASS
Quote 15/51 | seed=8 | PASS
Quote 16/51 | seed=8 | PASS
Quote 17/51 | seed=9 | PASS
Quote 18/51 | seed=9 | PASS
Quote 19/51 | seed=10 | PASS
Quote 20/51 | seed=10 | PASS
Quote 21/51 | seed=10 | PASS
Quote 22/51 | seed=11 | PASS
Quote 23/51 | seed=12 | PASS
Quote 24/51 | seed=12 | PASS
Quote 25/51 | seed=12 | PASS
Quote 26/51 | seed=13 | PASS
Quote 27/51 | seed=13 | PASS
Quote 28/51 | seed=13 | PASS
Quote 29/51 | seed=14 | PASS
Quote 30/51 | seed=15 | PASS
Quote 31/51 | seed=16 | PASS
Quote 32/51 | seed=16 | PASS
Quote 33/51 | seed=16 | PASS
Quote 34/51 | seed=17 | PA

<div style="direction: rtl; line-height: 1.8;">
<h2>۵. ساخت آزمایش غیر‌فعال و صف بازبینی</h2>
<p>رکورد آزمایشی تنها زمانی ساخته می‌شود که پاسخ span دقیق quote باشد و سؤال تمام کنترل‌های ماشینی را بگذراند. source_evidence رکورد جدید خودِ quote واقعی است؛ evidence قدیمی فقط با نام legacy_seed_evidence برای ممیزی نگهداری می‌شود.</p>
</div>


In [10]:
# Code Cell 10
experiment = []
review_queue = list(source_review_queue)
question_counts = Counter()

for item in generation:
    payload = item.get("payload") or {}

    if item["errors"]:
        continue

    question_counts[canon(payload["question"])] += 1

child_chunk_ids = {chunk["chunk_id"] for chunk in children}
medium_chunk_ids = {chunk["chunk_id"] for chunk in medium}

for item in generation:
    payload = item.get("payload") or {}
    errors = list(item["errors"])

    if (
        not errors
        and question_counts[canon(payload["question"])] > 1
    ):
        errors.append("duplicate_question")

    if errors:
        review_queue.append(
            {
                "quote_id": item["quote_id"],
                "source_seed_id": item["source_seed_id"],
                "quote": item["quote"],
                "question": payload.get("question"),
                "answer_span": payload.get("answer_span"),
                "review_reasons": sorted(set(errors)),
                "human_decision": None,
                "human_notes": None,
            }
        )
        continue

    seed = seeds[item["source_seed_id"]]
    candidate_id = str(
        uuid.uuid5(
            uuid.NAMESPACE_URL,
            (
                f"{build_fingerprint}:{item['quote_hash']}:"
                f"{canon(payload['question'])}"
            ),
        )
    )

    if item["chunk_id"] in child_chunk_ids:
        retrieval_candidate = "parent_child_children"
    elif item["chunk_id"] in medium_chunk_ids:
        retrieval_candidate = "structure_aware_medium"
    else:
        raise AssertionError(
            f"Unknown frozen chunk label: {item['chunk_id']}"
        )

    experiment.append(
        {
            "candidate_id": candidate_id,
            "dataset_version": (
                "golden_dataset_quote_grounded_experiment"
            ),
            "source_seed_id": item["source_seed_id"],
            "source_lineage": seed["lineage"],
            "question": payload["question"].strip(),
            "expected_answer": payload["answer_span"].strip(),
            "question_intent": payload["intent"],
            "evaluation_split": split_by_seed[item["source_seed_id"]],
            "source_evidence": item["quote"],
            "evidence_quote": item["quote"],
            "legacy_seed_evidence": seed["approved_evidence"],
            "taxonomy_topics": seed["taxonomy_topics"],
            "competency_ids": seed["competency_ids"],
            "pilot_case_id": provenance[
                item["source_seed_id"]
            ]["pilot_case_id"],
            "provenance_source": provenance[
                item["source_seed_id"]
            ]["source"],
            "document_ids": provenance[
                item["source_seed_id"]
            ]["document_ids"],
            "retrieval_labels": {
                "candidate": retrieval_candidate,
                "positive_chunk_ids": [item["chunk_id"]],
                "positive_parent_chunk_ids": (
                    [item["parent_chunk_id"]]
                    if item["parent_chunk_id"]
                    else []
                ),
                "label_basis": item["label_basis"],
            },
            "generation": {
                "model": chat_model,
                "prompt_version": QUESTION_PROMPT_VERSION,
                "build_fingerprint": build_fingerprint,
            },
            "generation_status": "EXPERIMENT_MACHINE_VALIDATED",
        }
    )

assert len(
    {record["candidate_id"] for record in experiment}
) == len(experiment)
assert len(
    {canon(record["question"]) for record in experiment}
) == len(experiment)

audit_size = max(1, math.ceil(len(experiment) * 0.10))
audit_records = sorted(
    experiment,
    key=lambda record: hashlib.sha256(
        ("audit:" + record["candidate_id"]).encode("utf-8")
    ).hexdigest(),
)[:audit_size]

for record in audit_records:
    review_queue.append(
        {
            "candidate_id": record["candidate_id"],
            "source_seed_id": record["source_seed_id"],
            "quote": record["evidence_quote"],
            "question": record["question"],
            "answer_span": record["expected_answer"],
            "review_reasons": ["random_quality_audit"],
            "human_decision": None,
            "human_notes": None,
        }
    )

experiment_path = golden_dir / f"{EXPERIMENT_NAME}.jsonl"
review_path = golden_dir / f"{EXPERIMENT_NAME}_review_queue.jsonl"
write_jsonl_atomic(experiment_path, experiment)
write_jsonl_atomic(review_path, review_queue)

print("Final quote candidates:", len(all_quote_ledger))
print("Machine-valid experiment records:", len(experiment))
print("Source-review records:", len(source_review_queue))
print(
    "Generation-review records:",
    sum(bool(item["errors"]) for item in generation),
)
print("Random audit records:", len(audit_records))
print("Total review queue records:", len(review_queue))
print("Experiment:", experiment_path)
print("\nNON-ACTIVE EXPERIMENT ASSEMBLY: PASS")


Final quote candidates: 51
Machine-valid experiment records: 51
Source-review records: 2
Generation-review records: 0
Random audit records: 6
Total review queue records: 8
Experiment: /Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/RAG_finance/notebooks/phase2/golden_dataset/golden_dataset_quote_grounded_experiment.jsonl

NON-ACTIVE EXPERIMENT ASSEMBLY: PASS


<div style="direction: rtl; line-height: 1.8;">
<h2>۶. diagnostic بازیابی</h2>
<p>این سنجش فقط یک diagnostic داخلی برای بررسی سازگاری سؤال با chunk برچسب‌خورده است. نتیجهٔ آن معادل benchmark نهایی NB13 نیست و rank برای حذف یا پذیرش رکورد استفاده نمی‌شود.</p>
</div>


In [11]:
# Code Cell 11
assert importlib.util.find_spec("sklearn") is not None
from sklearn.feature_extraction.text import TfidfVectorizer

child_ids = [chunk["chunk_id"] for chunk in children]
child_index_by_id = {
    chunk_id: index
    for index, chunk_id in enumerate(child_ids)
}
child_texts = [canon(chunk["content"]) for chunk in children]

vectorizer = TfidfVectorizer(
    tokenizer=words,
    token_pattern=None,
    ngram_range=(1, 2),
    min_df=1,
    sublinear_tf=True,
    norm="l2",
)
passage_matrix = vectorizer.fit_transform(child_texts)
diagnostic_rows = []

for record in experiment:
    labels = record["retrieval_labels"]

    if labels["candidate"] != "parent_child_children":
        diagnostic_rows.append(
            {
                "candidate_id": record["candidate_id"],
                "source_seed_id": record["source_seed_id"],
                "first_positive_rank": None,
                "excluded_reason": "no_parent_child_label",
            }
        )
        continue

    positive_chunk_id = labels["positive_chunk_ids"][0]
    assert positive_chunk_id in child_index_by_id

    query_vector = vectorizer.transform(
        [canon(record["question"])]
    )
    scores = (passage_matrix @ query_vector.T).toarray().ravel()
    ranking = np.argsort(-scores, kind="stable")
    positive_index = child_index_by_id[positive_chunk_id]
    rank = int(np.where(ranking == positive_index)[0][0]) + 1

    diagnostic_rows.append(
        {
            "candidate_id": record["candidate_id"],
            "source_seed_id": record["source_seed_id"],
            "first_positive_rank": rank,
            "hit_at_1": rank <= 1,
            "hit_at_3": rank <= 3,
            "hit_at_5": rank <= 5,
            "hit_at_10": rank <= 10,
            "reciprocal_rank": 1 / rank,
        }
    )

eligible_diagnostics = [
    row
    for row in diagnostic_rows
    if row["first_positive_rank"] is not None
]
assert eligible_diagnostics, "No parent-child diagnostics are available."

metrics = {
    "eligible_queries": len(eligible_diagnostics),
    "recall_at_1": float(
        np.mean([row["hit_at_1"] for row in eligible_diagnostics])
    ),
    "recall_at_3": float(
        np.mean([row["hit_at_3"] for row in eligible_diagnostics])
    ),
    "recall_at_5": float(
        np.mean([row["hit_at_5"] for row in eligible_diagnostics])
    ),
    "recall_at_10": float(
        np.mean([row["hit_at_10"] for row in eligible_diagnostics])
    ),
    "mrr": float(
        np.mean(
            [row["reciprocal_rank"] for row in eligible_diagnostics]
        )
    ),
}

print("=== LEXICAL RETRIEVAL DIAGNOSTIC ===")

for metric_name, metric_value in metrics.items():
    print(metric_name, ":", metric_value)

print("Rank was not used as an acceptance criterion.")
print("This diagnostic is not the final NB13 benchmark.")
print("\nRETRIEVAL DIAGNOSTIC: PASS")


=== LEXICAL RETRIEVAL DIAGNOSTIC ===
eligible_queries : 48
recall_at_1 : 0.6666666666666666
recall_at_3 : 0.7916666666666666
recall_at_5 : 0.8333333333333334
recall_at_10 : 0.8958333333333334
mrr : 0.7357520986606301
Rank was not used as an acceptance criterion.
This diagnostic is not the final NB13 benchmark.

RETRIEVAL DIAGNOSTIC: PASS


<div style="direction: rtl; line-height: 1.8;">
<h2>۷. کنترل نهایی artifactها و handoff</h2>
<p>در پایان، تمام رکوردهای آزمایشی دوباره از نظر span پاسخ، اتصال به chunk frozen، یکتایی و عدم ساخت نسخهٔ فعال کنترل می‌شوند.</p>
</div>


In [12]:
# Code Cell 12
current_v1_sha256 = sha256_file(v1_path)
assert current_v1_sha256 == dataset_sha256, (
    "Golden Dataset v1 changed during execution."
)

all_frozen_chunk_ids = child_chunk_ids | medium_chunk_ids

for record in experiment:
    assert record["source_evidence"] == record["evidence_quote"]
    assert canon(record["expected_answer"]) in canon(
        record["source_evidence"]
    )
    assert record["question"].endswith("؟")
    assert record["retrieval_labels"]["positive_chunk_ids"]
    assert set(
        record["retrieval_labels"]["positive_chunk_ids"]
    ) <= all_frozen_chunk_ids

assert len(read_jsonl(experiment_path)) == len(experiment)
assert len(read_jsonl(review_path)) == len(review_queue)

active_version_pattern = re.compile(r"^golden_dataset_v\d+\.jsonl$")
assert not active_version_pattern.match(experiment_path.name)
assert experiment_path.name != SOURCE_DATASET_FILENAME

report = {
    "schema_version": (
        "golden_dataset_quote_grounded_experiment_report_v2"
    ),
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "build_fingerprint": build_fingerprint,
    "source": {
        "dataset": str(v1_path.relative_to(project_root)),
        "sha256": dataset_sha256,
        "records": len(v1),
        "legacy_questions_used": False,
        "source_dataset_modified": False,
    },
    "experiment": {
        "path": str(experiment_path.relative_to(project_root)),
        "sha256": sha256_file(experiment_path),
        "records": len(experiment),
        "quote_candidates": len(all_quote_ledger),
        "represented_seed_ids": sorted(
            {record["source_seed_id"] for record in experiment}
        ),
        "active_dataset_created": False,
    },
    "quality": {
        "initial_quote_count": len(quote_ledger),
        "window_repair_quote_count": len(resolved_quote_windows),
        "manual_source_review_seed_ids": sorted(
            {item["source_seed_id"] for item in source_review_queue}
        ),
        "generation_review_count": sum(
            bool(item["errors"]) for item in generation
        ),
        "random_audit_count": len(audit_records),
        "review_queue_records": len(review_queue),
        "exact_answer_span_required": True,
        "source_evidence_is_exact_quote": True,
    },
    "cache": {
        "namespace": cache_namespace,
        "successful_records_reused": cache_reuse_count,
        "live_requests": live_request_count,
        "notebook_filename_affects_cache": False,
    },
    "retrieval_diagnostic": {
        "model": "persian_tfidf_lexical",
        "candidate": "parent_child_children",
        "metrics": metrics,
        "scope": "INTERNAL_LABEL_CONSISTENCY_DIAGNOSTIC",
        "selection_bias_guard": "RANK_NOT_USED_FOR_ACCEPTANCE",
    },
    "status": "EXPERIMENT_COMPLETE_NOT_A_GOLDEN_VERSION",
    "production_approved": False,
}

report_path = golden_dir / f"{EXPERIMENT_NAME}_quality_report.json"
diagnostic_path = (
    golden_dir / f"{EXPERIMENT_NAME}_retrieval_diagnostic.json"
)
manifest_output_path = (
    golden_dir / f"{EXPERIMENT_NAME}_build_manifest.json"
)

write_json_atomic(report_path, report)
write_json_atomic(
    diagnostic_path,
    {
        "metrics": metrics,
        "rows": diagnostic_rows,
        "scope": "INTERNAL_LABEL_CONSISTENCY_DIAGNOSTIC",
    },
)
write_json_atomic(
    manifest_output_path,
    {
        **build_identity,
        "build_fingerprint": build_fingerprint,
        "experiment_path": str(
            experiment_path.relative_to(project_root)
        ),
        "experiment_sha256": sha256_file(experiment_path),
        "creates_active_dataset": False,
    },
)

required_outputs = (
    experiment_path,
    review_path,
    report_path,
    diagnostic_path,
    manifest_output_path,
)
assert all(path.is_file() for path in required_outputs)

print("=== NB12G CLEAN FINAL HANDOFF ===")
print("Status:", report["status"])
print("Active v1 unchanged:", current_v1_sha256 == dataset_sha256)
print("Active dataset created:", False)
print("Experiment records:", len(experiment))
print("Review queue records:", len(review_queue))
print(
    "Manual source-review seeds:",
    report["quality"]["manual_source_review_seed_ids"],
)
print("Notebook filename affects cache:", False)
print("Experiment:", experiment_path)
print("Review queue:", review_path)
print("Quality report:", report_path)
print("\nNB12G CLEAN FINAL HANDOFF: PASS")


=== NB12G CLEAN FINAL HANDOFF ===
Status: EXPERIMENT_COMPLETE_NOT_A_GOLDEN_VERSION
Active v1 unchanged: True
Active dataset created: False
Experiment records: 51
Review queue records: 8
Manual source-review seeds: [7, 22]
Notebook filename affects cache: False
Experiment: /Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/RAG_finance/notebooks/phase2/golden_dataset/golden_dataset_quote_grounded_experiment.jsonl
Review queue: /Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/RAG_finance/notebooks/phase2/golden_dataset/golden_dataset_quote_grounded_experiment_review_queue.jsonl
Quality report: /Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/RAG_finance/notebooks/phase2/golden_dataset/golden_dataset_quote_grounded_experiment_quality_report.json

NB12G CLEAN FINAL HANDOFF: PASS


<div style="direction: rtl; line-height: 1.8;">
<h2>مرز مرحلهٔ بعد</h2>
<p>این notebook هیچ Golden Dataset v2 نمی‌سازد و NB13 را اجرا نمی‌کند. خروجی فعلی فقط یک آزمایش quote-grounded است. seedهای باقی‌مانده در صف review و نمونه‌های audit باید بررسی شوند. پس از بازخورد مؤلف، ساخت v2 در یک مرحلهٔ مستقل و مقایسهٔ صریح v1، experiment و v2 انجام خواهد شد.</p>
</div>
